# AIDev Dataset Download

This utility notebook downloads the six raw AIDev parquet tables from HuggingFace into your Google Drive. It is only needed if you want to reproduce the full preprocessing pipeline from scratch.

**For reviewers, this notebook is not required.** The preprocessed study population (`data/processed/pr_closed.parquet`) and the code diff table (`data/raw/pr_commit_details.parquet`) are already provided in the shared Drive folder and are sufficient to run all model notebooks.

| Item | Description |
|------|-------|
| Source | `hao-li/AIDev` on HuggingFace |
| Pinned commit | `512e07014b7b6e34cc1080372caa1c2bc054369d` |
| Coverage | December 2024 to July 31, 2025 |
| Tables downloaded | pull_request, repository, pr_commits, pr_commit_details, pr_task_type, related_issue |
| Output location | `data/raw/` under your DRIVE_BASE folder |
| Runtime | ~10–15 min depending on connection speed |

Once this notebook completes, run `multimodal_lr.ipynb` through the preprocessing cell to regenerate `data/processed/pr_closed.parquet` from the raw tables.

In [ ]:
# Run this notebook ONCE to download the raw AIDev tables to
# your Google Drive.

from google.colab import drive
from pathlib import Path
import subprocess, sys

drive.mount('/content/drive')
print('Drive mounted')

# ---------------------------------------------------------------
# CONFIGURE THIS PATH before running any other cell.
# Set DRIVE_BASE to the folder on your Google Drive where you
# want to place the project. Must match DRIVE_BASE in the other
# notebooks.
# Example: Path('/content/drive/MyDrive/AgenticPRRejection')
# ---------------------------------------------------------------
DRIVE_BASE = Path('/content/drive/MyDrive/Thesis/AgenticPRRejection')
print(f'DRIVE_BASE: {DRIVE_BASE}')
print(f'Exists: {DRIVE_BASE.exists()}')

DRIVE_DIR = DRIVE_BASE / 'data' / 'raw'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Target directory: {DRIVE_DIR}')

Mounted at /content/drive
Drive mounted
DRIVE_BASE: /content/drive/MyDrive/Thesis/AgenticPRRejection
Exists: True
Target directory: /content/drive/MyDrive/Thesis/AgenticPRRejection/data/raw


In [ ]:
# Environment info
import sys
import pandas as pd
import huggingface_hub

print(f'Python          : {sys.version}')
print(f'pandas          : {pd.__version__}')
print(f'huggingface_hub : {huggingface_hub.__version__}')

Python          : 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
pandas          : 2.2.2
huggingface_hub : 1.11.0


In [ ]:
# Download all AIDev tables from HuggingFace
# Pinned to commit 512e07014b7b6e34cc1080372caa1c2bc054369d
# to reproduce the exact dataset version used in this study
# (December 2024 to July 2025).
# Skip any table that already exists on Drive.

import shutil
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'hao-li/AIDev'
HF_COMMIT  = '512e07014b7b6e34cc1080372caa1c2bc054369d'

TABLE_NAMES = [
    'pull_request',
    'repository',
    'pr_commits',
    'pr_commit_details',
    'pr_task_type',
    'related_issue',
]

for name in TABLE_NAMES:
    dest = DRIVE_DIR / f'{name}.parquet'
    if dest.exists():
        print(f'  {name}: already exists, skipping.')
        continue
    print(f'Downloading {name}...')
    tmp_path = hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=f'{name}.parquet',
        repo_type='dataset',
        revision=HF_COMMIT,
        local_dir='/content/tmp_aidev',
    )
    shutil.copy2(tmp_path, dest)
    print(f'  Saved: {dest} ({dest.stat().st_size / 1e6:.1f} MB)')

print('\nAll tables downloaded.')

  pull_request: already exists, skipping.
  repository: already exists, skipping.
  pr_commits: already exists, skipping.
  pr_commit_details: already exists, skipping.
  pr_task_type: already exists, skipping.
  related_issue: already exists, skipping.

All tables downloaded.


In [ ]:
# Verify all tables
import pandas as pd

print('Verifying tables...\n')
for name in TABLE_NAMES:
    path = DRIVE_DIR / f'{name}.parquet'
    df = pd.read_parquet(path)
    print(f'{name}: {df.shape}, {path.stat().st_size / 1e6:.1f} MB on disk')

Verifying tables...

pull_request: (33596, 14), 16.9 MB on disk
repository: (2807, 7), 0.2 MB on disk
pr_commits: (88576, 5), 8.3 MB on disk
pr_commit_details: (711923, 14), 485.2 MB on disk
pr_task_type: (33596, 6), 3.0 MB on disk
related_issue: (4923, 3), 0.1 MB on disk


In [ ]:
# Clean up Colab temp files
import shutil as _shutil
_shutil.rmtree('/content/tmp_aidev', ignore_errors=True)
print('Temp files removed.')
print(f'\nFiles saved to {DRIVE_DIR}:')
for f in sorted(DRIVE_DIR.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

Temp files removed.

Files saved to /content/drive/MyDrive/Thesis/AgenticPRRejection/data/raw:
  .cache  (0.0 MB)
  pr_commit_details.parquet  (485.2 MB)
  pr_commits.parquet  (8.3 MB)
  pr_task_type.parquet  (3.0 MB)
  pull_request.parquet  (16.9 MB)
  related_issue.parquet  (0.1 MB)
  repository.parquet  (0.2 MB)
  train_checkpoint.npz  (22.9 MB)


In [ ]:
# Inspect pull_request
import pyarrow.parquet as pq
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/Thesis/AgenticPRRejection/data/raw')

display(pq.read_table(DRIVE_DIR / 'pull_request.parquet').slice(0, 5).to_pandas())

,id,number,title,body,agent,user_id,user,state,created_at,closed_at,merged_at,repo_id,repo_url,html_url
0,3264933329,2911,Fix: Wait for all partitions in load_collectio...,## Summary\n\nFixes an issue where `load_colle...,Claude_Code,108661493,weiliu1031,closed,2025-07-26T02:59:01Z,2025-07-29T07:01:20Z,None,191751505,https://api.github.com/repos/milvus-io/pymilvus,https://github.com/milvus-io/pymilvus/pull/2911
1,3265118634,2,ファイルパス参照を相対パスに統一し、doc/からdocs/に統一,## 背景\n\n現在、本プロジェクトにおいて以下のパス構成の不整合が生じています：\n\n...,Claude_Code,61827001,cm-kojimat,closed,2025-07-26T04:56:55Z,2025-07-26T22:12:24Z,2025-07-26T22:12:24Z,1025472321,https://api.github.com/repos/classmethod/tsumiki,https://github.com/classmethod/tsumiki/pull/2
2,3265640341,30,Add build staleness detection for debug CLI,## Summary\r\n\r\n Implements comprehensive b...,Claude_Code,7475,MSch,closed,2025-07-26T13:31:19Z,2025-07-26T13:37:22Z,2025-07-26T13:37:22Z,988488798,https://api.github.com/repos/steipete/Peekaboo,https://github.com/steipete/Peekaboo/pull/30
3,3265709660,205,feat: add comprehensive README screenshots wit...,## Type of Change\n\n- [ ] 🐛 `bug` - Bug fix (...,Claude_Code,80381,sugyan,closed,2025-07-26T14:07:22Z,2025-07-26T14:45:30Z,2025-07-26T14:45:30Z,999285986,https://api.github.com/repos/sugyan/claude-cod...,https://github.com/sugyan/claude-code-webui/pu...
4,3265782173,17625,chore: remove HashedPostStateProvider trait,## Summary\r\n\r\n#17545 \r\n\r\nRemove the un...,Claude_Code,47593288,adust09,open,2025-07-26T15:02:48Z,None,None,537233603,https://api.github.com/repos/paradigmxyz/reth,https://github.com/paradigmxyz/reth/pull/17625


In [ ]:
# Inspect repository
display(pq.read_table(DRIVE_DIR / 'repository.parquet').slice(0, 5).to_pandas())

,id,url,license,full_name,language,forks,stars
0,966235850,https://api.github.com/repos/kizuna-ai-lab/sokuji,AGPL-3.0,kizuna-ai-lab/sokuji,TypeScript,11,254
1,386644013,https://api.github.com/repos/freenet/freenet-core,NOASSERTION,freenet/freenet-core,Rust,94,2391
2,977155585,https://api.github.com/repos/coleam00/mcp-craw...,MIT,coleam00/mcp-crawl4ai-rag,Python,474,1447
3,528194129,https://api.github.com/repos/vexxhost/atmosphere,None,vexxhost/atmosphere,Smarty,34,142
4,703998226,https://api.github.com/repos/JonasKruckenberg/k23,Apache-2.0,JonasKruckenberg/k23,Rust,32,515


In [ ]:
# Inspect pr_commits
display(pq.read_table(DRIVE_DIR / 'pr_commits.parquet').slice(0, 5).to_pandas())

,sha,pr_id,author,committer,message
0,15a543882cbad0c9348640cffdd71fb71ac34953,3205734508,rubys,rubys,refactor: Convert Build.Compose from string to...
1,cdd9bab35891037692f1d201db37d34d48342332,3107321792,haasonsaas,haasonsaas,Fix Windows compatibility issues in shell exec...
2,219323e2aa1dacd76a422ad24cabf6976c9a6ac0,3107321792,haasonsaas,haasonsaas,Add Windows path validation fixes and comprehe...
3,58c52841e96e9fdcb70befadf7b3994519834861,3107321792,haasonsaas,haasonsaas,Address CodeRabbit review feedback - fix criti...
4,b04470fe6bae5e6802f1fe83e7c5f0b9273e74a3,3107321792,haasonsaas,haasonsaas,Fix Windows compatibility tests - address patt...


In [ ]:
# Inspect pr_commit_details (first 5 rows only)
display(pq.read_table(DRIVE_DIR / 'pr_commit_details.parquet').slice(0, 5).to_pandas())

,sha,pr_id,author,committer,message,commit_stats_total,commit_stats_additions,commit_stats_deletions,filename,status,additions,deletions,changes,patch
0,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,.github/workflows/compile-llms-txt.yml,added,38.0,0.0,38.0,"@@ -0,0 +1,38 @@\n+name: Compile llms.txt\n+\n..."
1,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,docs/compile_llms_txt.py,added,47.0,0.0,47.0,"@@ -0,0 +1,47 @@\n+import os\n+from pathlib im..."
2,2f9d54dda4f0c87c19e0bbeb9936f525d0587e16,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Add llms.txt compilation system for AI model d...,23008,23008,0,llms.txt,added,22923.0,0.0,22923.0,None
3,dbd1b5f129f7cffa5ce284d7255814c98bcc38a2,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Fix lint issues: remove unused variable and ap...,35,18,17,docs/compile_llms_txt.py,modified,18.0,17.0,35.0,"@@ -1,47 +1,48 @@\n import os\n from pathlib i..."
4,c2659cfdedf666c8f14753d71664563c2a932b23,3271196926,devin-ai-integration[bot],devin-ai-integration[bot],Update llms.txt to follow official standard wi...,23035,89,22946,docs/compile_llms_txt.py,modified,51.0,36.0,87.0,"@@ -3,45 +3,60 @@\n \n \n def compile_llms_txt..."


In [ ]:
# Inspect pr_task_type
display(pq.read_table(DRIVE_DIR / 'pr_task_type.parquet').slice(0, 5).to_pandas())

,agent,id,title,reason,type,confidence
0,Claude_Code,3264933329,Fix: Wait for all partitions in load_collectio...,title provides conventional commit label,fix,10
1,Claude_Code,3265709660,feat: add comprehensive README screenshots wit...,title provides conventional commit label,feat,10
2,Claude_Code,3265782173,chore: remove HashedPostStateProvider trait,title provides conventional commit label,chore,10
3,Claude_Code,3231949586,feat(swagger): Add Swagger annotations to Batc...,title provides conventional commit label,feat,10
4,Claude_Code,3231950376,feat(swagger): Add Swagger annotations to Batc...,title provides conventional commit label,feat,10


In [ ]:
# Inspect related_issue
display(pq.read_table(DRIVE_DIR / 'related_issue.parquet').slice(0, 5).to_pandas())

,pr_id,issue_id,source
0,3234102722,3.233113e+09,body
1,3214782537,3.189853e+09,body
2,3164567685,3.164539e+09,body
3,3165176791,2.510821e+09,body
4,3165431934,3.165438e+09,body
